# HeLa raw images to Ultrack tracks

This notebook develops the raw-image workflow interactively on a small subset of `Fluo-N2DL-HeLa`. It keeps the main Ultrack stages separate so that the foreground and contour inputs can be checked before linking and solving.

Initial subset: sequence `01`, frames 12–16. Frame 12 has a gold-segmentation annotation, making it useful for visual inspection. Once the workflow is satisfactory, use a new run name and expand to the complete sequence.

The initial segmentation baseline uses Ultrack's `detect_foreground` and `robust_invert` image-processing utilities. This verifies that the pipeline begins with raw microscopy images, but it is not a direct reproduction of the published Cellpose, StarDist, watershed, or ensemble results.


## Environment and paths

The notebook reads shared data in place and writes generated artifacts only to the personal Ultrack run directory.

Set these variables before starting the VS Code notebook kernel, or set them in a temporary notebook cell before running the configuration cell:

```bash
export CTC_ROOT=/path/to/shared/ctc
export ULTRACK_RUN_ROOT=/path/to/personal/runs/ultrack
```


In [ ]:
import os
import sys
from importlib.metadata import version
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile
from ultrack import MainConfig, Tracker
from ultrack.imgproc import detect_foreground, robust_invert

print(f"Python: {sys.version.split()[0]}")
print(f"Ultrack: {version('ultrack')}")
print(f"Python executable: {sys.executable}")


In [ ]:
ctc_root_value = os.environ.get("CTC_ROOT")
run_root_value = os.environ.get("ULTRACK_RUN_ROOT")

if not ctc_root_value:
    raise RuntimeError("Set CTC_ROOT before running this notebook.")
if not run_root_value:
    raise RuntimeError("Set ULTRACK_RUN_ROOT before running this notebook.")

CTC_ROOT = Path(ctc_root_value).expanduser()
ULTRACK_RUN_ROOT = Path(run_root_value).expanduser()

if not CTC_ROOT.is_dir():
    raise FileNotFoundError(f"CTC root does not exist: {CTC_ROOT}")

print(f"CTC root: {CTC_ROOT}")
print(f"Ultrack run root: {ULTRACK_RUN_ROOT}")


## 2. Choose the dataset subset and segmentation parameters

`FRAME_STOP` is exclusive, following normal Python slicing. Change `RUN_NAME` whenever the subset or parameters change so that separate experiments are not mixed.


In [ ]:
DATASET = "Fluo-N2DL-HeLa"
SEQUENCE = "01"
FRAME_START = 12
FRAME_STOP = 17

FOREGROUND_SIGMA = 15.0
REMOVE_HIST_MODE = False
MIN_FOREGROUND = 0.0
CONTOUR_SIGMA = 1.0

RUN_NAME = "hela-seq01-t012-016-auto-fg15-contour1"

DATASET_ROOT = CTC_ROOT / DATASET
RAW_DIR = DATASET_ROOT / SEQUENCE
GT_SEG_DIR = DATASET_ROOT / f"{SEQUENCE}_GT/SEG"
RUN_DIR = ULTRACK_RUN_ROOT / RUN_NAME

print(f"Raw input: {RAW_DIR}")
print(f"Sparse GT reference: {GT_SEG_DIR}")
print(f"Run output: {RUN_DIR}")


## 3. Load raw microscopy images

Only files from the raw sequence folder are loaded. Nothing under `GT`, `ST`, or `ERR_SEG` is used to construct the segmentation inputs.


In [ ]:
all_raw_paths = sorted(RAW_DIR.glob("t*.tif"))
raw_paths = all_raw_paths[FRAME_START:FRAME_STOP]

if len(raw_paths) != FRAME_STOP - FRAME_START:
    raise ValueError(
        f"Expected {FRAME_STOP - FRAME_START} raw frames, "
        f"but found {len(raw_paths)}."
    )

raw_images = np.stack([tifffile.imread(path) for path in raw_paths])
frame_numbers = [int(path.stem.removeprefix("t")) for path in raw_paths]

print(f"Files: {raw_paths[0].name} to {raw_paths[-1].name}")
print(f"Array shape: {raw_images.shape}")
print(f"Data type: {raw_images.dtype}")
print(f"Intensity range: {raw_images.min()} to {raw_images.max()}")


In [ ]:
DISPLAY_LOCAL_INDEX = 0
display_frame = frame_numbers[DISPLAY_LOCAL_INDEX]

plt.figure(figsize=(8, 5))
plt.imshow(raw_images[DISPLAY_LOCAL_INDEX], cmap="gray")
plt.title(f"{DATASET} sequence {SEQUENCE}, frame {display_frame}")
plt.axis("off")
plt.show()


## 4. Create foreground and contour maps from raw images

Ultrack requires two canonical inputs with the same `(time, y, x)` shape:

- `foreground`: where cells may be present;
- `contours`: confidence that pixels belong to cell boundaries.

Each frame is processed independently so the Gaussian filters do not smooth across time. Ground truth is not used in either calculation.


In [ ]:
foreground = np.stack(
    [
        np.asarray(
            detect_foreground(
                frame,
                sigma=FOREGROUND_SIGMA,
                remove_hist_mode=REMOVE_HIST_MODE,
                min_foreground=MIN_FOREGROUND,
            )
        )
        for frame in raw_images
    ]
).astype(bool)

contours = np.stack(
    [
        np.asarray(robust_invert(frame, sigma=CONTOUR_SIGMA))
        for frame in raw_images
    ]
).astype(np.float32)

if foreground.shape != raw_images.shape or contours.shape != raw_images.shape:
    raise ValueError("Raw, foreground, and contour shapes must match.")

print(f"Foreground shape and dtype: {foreground.shape}, {foreground.dtype}")
print(f"Contour shape and dtype: {contours.shape}, {contours.dtype}")


In [ ]:
segmentation_input_summary = pd.DataFrame(
    {
        "frame": frame_numbers,
        "foreground fraction": foreground.mean(axis=(1, 2)),
        "contour minimum": contours.min(axis=(1, 2)),
        "contour maximum": contours.max(axis=(1, 2)),
        "contour mean": contours.mean(axis=(1, 2)),
    }
)
segmentation_input_summary


In [ ]:
image = raw_images[DISPLAY_LOCAL_INDEX]
mask = foreground[DISPLAY_LOCAL_INDEX]
contour = contours[DISPLAY_LOCAL_INDEX]

figure, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(image, cmap="gray")
axes[0].set_title("Raw image")
axes[1].imshow(mask, cmap="gray")
axes[1].set_title("Detected foreground")
axes[2].imshow(contour, cmap="magma")
axes[2].set_title("Contour confidence")
axes[3].imshow(image, cmap="gray")
axes[3].imshow(np.ma.masked_where(~mask, mask), cmap="spring", alpha=0.35)
axes[3].set_title("Foreground over raw")

for axis in axes:
    axis.axis("off")

figure.suptitle(f"Frame {display_frame}")
plt.tight_layout()
plt.show()


## 5. Compare with sparse gold segmentation

Gold segmentation is used only to inspect the foreground map. It is not passed to Ultrack. Because the gold mask contains selected manually annotated cells rather than exhaustive background labels, the notebook reports the fraction of annotated cell pixels captured by the foreground but does not treat every unlabeled pixel as true background.


In [ ]:
gt_path = GT_SEG_DIR / f"man_seg{display_frame:03d}.tif"

if gt_path.is_file():
    gt_labels = tifffile.imread(gt_path)
    gt_mask = gt_labels > 0
    annotated_pixel_coverage = mask[gt_mask].mean()

    figure, axes = plt.subplots(1, 3, figsize=(14, 5))
    axes[0].imshow(image, cmap="gray")
    axes[0].set_title("Raw image")
    axes[1].imshow(image, cmap="gray")
    axes[1].imshow(
        np.ma.masked_where(~gt_mask, gt_labels),
        cmap="tab20",
        alpha=0.6,
    )
    axes[1].set_title("Sparse gold segmentation")
    axes[2].imshow(image, cmap="gray")
    axes[2].imshow(
        np.ma.masked_where(~mask, mask),
        cmap="spring",
        alpha=0.35,
    )
    axes[2].set_title("Detected foreground")

    for axis in axes:
        axis.axis("off")

    plt.tight_layout()
    plt.show()
    print(
        "Fraction of annotated cell pixels inside foreground: "
        f"{annotated_pixel_coverage:.3f}"
    )
else:
    print(f"No gold-segmentation file exists for frame {display_frame}.")


### Inspection checkpoint

Before running Ultrack, verify visually that:

1. the foreground contains the nuclei of interest, including dim nuclei;
2. obvious background occupies as little foreground as practical; and
3. contour values are higher at plausible boundaries and lower inside cells.

If cells are missing, adjust the foreground parameters and rerun from Section 4. If the contour map is noisy or erases boundaries, adjust `CONTOUR_SIGMA`. Record every parameter set tested and do not choose parameters using final evaluation scores alone.


## 6. Configure Ultrack

The following cells run segmentation, linking, and solving separately. They use only the raw-derived foreground and contour maps. `overwrite=True` replaces the Ultrack database content inside this run directory, so change `RUN_NAME` before testing a different experiment that should be retained.


In [ ]:
RUN_DIR.mkdir(parents=True, exist_ok=True)

config = MainConfig()
config.data_config.working_dir = RUN_DIR
config.tracking_config.solver_name = "GUROBI"

print(config)


## 7. Create segmentation hypotheses

Run this cell only after the inspection checkpoint is satisfactory. This is Ultrack's segmentation stage; it creates candidate regions in the run database.


In [ ]:
tracker = Tracker(config=config)
tracker.segment(
    foreground=foreground,
    contours=contours,
    overwrite=True,
)
print("Segmentation stage complete.")


## 8. Link candidate segments


In [ ]:
tracker.link()
print("Linking stage complete.")


## 9. Solve the tracking problem

This stage uses the configured Gurobi solver to select a temporally consistent set of segments and links.


In [ ]:
tracker.solve()
print("Solving stage complete.")


## 10. Export and inspect the subset result


In [ ]:
tracks, graph = tracker.to_tracks_layer()
tracks_path = RUN_DIR / "tracks.csv"
segments_path = RUN_DIR / "segments.zarr"
ctc_output_dir = RUN_DIR / f"{SEQUENCE}_RES"

tracks.to_csv(tracks_path, index=False)
segments = tracker.to_zarr(
    store_or_path=segments_path,
    overwrite=True,
)
tracker.to_ctc(
    ctc_output_dir,
    first_frame=FRAME_START,
    overwrite=True,
)

print(f"Track rows: {len(tracks)}")
print(f"Unique tracks: {tracks['track_id'].nunique()}")
print(f"Lineage graph entries: {len(graph)}")
print(f"Tracks CSV: {tracks_path}")
print(f"Segments: {segments_path}")
print(f"CTC subset output: {ctc_output_dir}")


In [ ]:
result_labels = np.asarray(segments)

figure, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(image, cmap="gray")
axes[0].set_title("Raw image")
axes[1].imshow(result_labels[DISPLAY_LOCAL_INDEX], cmap="nipy_spectral")
axes[1].set_title("Ultrack result labels")
axes[2].imshow(image, cmap="gray")
axes[2].imshow(
    np.ma.masked_where(
        result_labels[DISPLAY_LOCAL_INDEX] == 0,
        result_labels[DISPLAY_LOCAL_INDEX],
    ),
    cmap="nipy_spectral",
    alpha=0.5,
)
axes[2].set_title("Ultrack labels over raw")

for axis in axes:
    axis.axis("off")

plt.tight_layout()
plt.show()


## 11. Record the experiment

Record the following in the Week 3 working log before changing parameters or expanding the frame range:

- dataset, sequence, and exact frame range;
- foreground and contour parameters;
- relevant Ultrack configuration changes;
- solver, CPU/memory allocation, and runtime;
- run directory and exported files;
- foreground coverage of annotated pixels; and
- visual successes, failures, and the next parameter change.

Do not treat this five-frame subset as the final HeLa benchmark run. The complete sequence must use a documented, fixed configuration and be evaluated separately. Clear bulky notebook outputs before committing.
